In [ ]:
!sudo apt-get update
!sudo apt-get install libopencv-dev

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
%%writefile opencv_stitcher.cpp
#include <iostream>
#include <vector>
#include <opencv2/opencv.hpp>
#include <opencv2/stitching.hpp>

// Use standard and OpenCV namespaces to avoid prefixing with std:: and cv::
using namespace cv;
using namespace std;

int main(int argc, char** argv) {
    // Verify that the program received exactly the expected number of arguments:
    // argv[0] = program name, argv[1] = input folder, argv[2] = output file
    if (argc < 3) {
        cout << "Usage: ./app <input_folder_path> <output_file_path>" << endl;
        return -1;
    }

    // Extract the input directory and desired output file path from command line arguments
    String folderPath = argv[1];
    String outputPath = argv[2];

    String path = folderPath + "/*.jpg";

    // 'result' will store the absolute file paths of all matching images
    vector<String> result;
    // 'raw_images' will hold the actual loaded image matrices for the stitcher
    vector<Mat> raw_images;

    // Use OpenCV's glob utility to find all files matching the pattern.
    // false = non-recursive search (only search the top level of the folder)
    glob(path, result, false);

    // Iterate through all the discovered file paths
    for (size_t k = 0; k < result.size(); ++k) {
        // Load the image into memory using imread
        Mat image = imread(result[k]);

        // Skip invalid or unreadable images (prevents crashing during processing)
        if (image.empty()) continue;

        // RAM Protection: Calculate a scale factor to prevent out-of-memory errors on limited hardware.
        // 1200.0 is the maximum allowed dimension (width or height).
        double scale = 1200.0 / max(image.cols, image.rows);

        // If the original image is larger than 1200 pixels on its longest side, scale it down
        if (scale < 1.0) {
            // Resize the image in-place based on the calculated scale ratio
            resize(image, image, Size(), scale, scale);
        }

        // Add the carefully processed/resized image to our main stitching collection
        raw_images.push_back(image);
    }

    // Inform the user about the initial processing status
    cout << "Processing folder: " << folderPath << endl;
    cout << "Number of input images: " << raw_images.size() << endl;

    // Halt execution if the glob search found no valid JPEG files
    if(raw_images.empty()) {
        cout << "Error: No images found in the directory." << endl;
        return -1;
    }

    // 'pano' will store the final assembled panorama image matrix
    Mat pano;

    // Initialize the built-in Stitcher module.
    // The 'PANORAMA' mode configures the stitcher for standard spherical warping and blending,
    // which is ideal for landscapes and large scene aggregations.
    Ptr<Stitcher> stitcher = Stitcher::create(Stitcher::PANORAMA);

    cout << "Stitching in progress. This may take a moment..." << endl;

    // Execute the main stitching pipeline: feature detection, matching, camera estimation,
    // warping, and seam blending. Returns a status enum indicating success or failure type.
    Stitcher::Status status = stitcher->stitch(raw_images, pano);

    // Evaluate the stitcher's exit status
    if (status == Stitcher::OK) {
        cout << "Final Panorama Generated Successfully!" << endl;
        // Save the resulting stitched matrix 'pano' mapped to the designated filesystem path.
        // Using imwrite avoids needing a GUI display window or HighGUI backend, ideal for headless environments.
        imwrite(outputPath, pano);
    } else {
        // Stitcher::OK is 0. Other failure codes indicate:
        // 1 = ERR_NEED_MORE_IMGS (Not enough overlapping features to form a sequence)
        // 2 = ERR_HOMOGRAPHY_EST_FAIL (Resolution mismatch or difficult geometry)
        // 3 = ERR_CAMERA_PARAMS_ADJUST_FAIL (Failed to solve camera orientation parameters)
        cout << "Error stitching. Error Code: " << int(status) << endl;
    }

    return 0;
}

Writing opencv_stitcher.cpp


In [17]:
!g++ opencv_stitcher.cpp -o app `pkg-config --cflags --libs opencv4`

In [18]:
# 1. Create the Output directory in your Drive if it doesn't exist yet
!mkdir -p "/content/drive/MyDrive/MachineVision/Output"

In [19]:
# 2. Run the application 3 times.
# Format: !./app "<Input_Folder_Path>" "<Output_File_Path>"
!./app "/content/drive/MyDrive/MachineVision/Data/office2" "/content/drive/MyDrive/MachineVision/Output/panorama_opencv_stitcher_1.jpg"


Processing folder: /content/drive/MyDrive/MachineVision/Data/office2
Number of input images: 29
Stitching in progress. This may take a moment...
Final Panorama Generated Successfully!


In [20]:
!./app "/content/drive/MyDrive/MachineVision/Data/WLH" "/content/drive/MyDrive/MachineVision/Output/panorama_opencv_stitcher_2.jpg"


Processing folder: /content/drive/MyDrive/MachineVision/Data/WLH
Number of input images: 25
Stitching in progress. This may take a moment...
Final Panorama Generated Successfully!


In [21]:
!./app "/content/drive/MyDrive/MachineVision/Data/StJames" "/content/drive/MyDrive/MachineVision/Output/panorama_opencv_stitcher_3.jpg"

Processing folder: /content/drive/MyDrive/MachineVision/Data/StJames
Number of input images: 16
Stitching in progress. This may take a moment...
Final Panorama Generated Successfully!


In [22]:
%%writefile advanced_stitcher.cpp
#include <iostream>
#include <vector>
#include <opencv2/opencv.hpp>
// Include internal detail headers for manual access to the stitching pipeline stages
#include <opencv2/stitching/detail/autocalib.hpp>
#include <opencv2/stitching/detail/blenders.hpp>
#include <opencv2/stitching/detail/camera.hpp>
#include <opencv2/stitching/detail/exposure_compensate.hpp>
#include <opencv2/stitching/detail/matchers.hpp>
#include <opencv2/stitching/detail/motion_estimators.hpp>
#include <opencv2/stitching/detail/seam_finders.hpp>
#include <opencv2/stitching/detail/warpers.hpp>

using namespace std;
using namespace cv;
// cv::detail contains the lower-level stitching components we are explicitly controlling
using namespace cv::detail;

int main(int argc, char** argv) {
    // Validate that we have the proper command-line arguments: folder, and output file
    if (argc < 3) {
        cout << "Usage: ./stitcher_app <input_folder> <output_file_path>" << endl;
        return -1;
    }

    String folderPath = argv[1];
    String outputPath = argv[2];
    vector<String> image_paths;

    // Use glob to discover all JPEG files in the specified input directory.
    // The false parameter indicates a non-recursive search.
    glob(folderPath + "/*.jpg", image_paths, false);

    vector<Mat> images;

    cout << "Loading and downscaling images for Colab compatibility..." << endl;
    for (const auto& path : image_paths) {
        // Load each image
        Mat img = imread(path);

        // Skip images that cannot be loaded to avoid crashing later operations
        if (img.empty()) continue;

        // ==========================================
        // COLAB FIX 1: AGGRESSIVE DOWNSCALING
        // ==========================================
        // Shrink images to a max dimension of 400px.
        // 30 full-res images will instantly crash Colab because the manual pipeline
        // stores multiple floating-point representations of warping masks and images in memory.
        double scale = 400.0 / max(img.cols, img.rows);
        if (scale < 1.0) {
            // Resize the image to enforce the RAM constraint
            resize(img, img, Size(), scale, scale);
        }
        images.push_back(img);
    }

    // A panorama requires a minimum of 2 images to be stitched
    if (images.size() < 2) {
        cout << "Error: Need at least 2 images to stitch." << endl;
        return -1;
    }

    cout << "Loaded " << images.size() << " images." << endl;

    // ==========================================
    // STEP 1: FEATURE DETECTION AND MATCHING
    // ==========================================
    cout << "Step 1: Finding and Matching Features (ORB)..." << endl;
    // features will store keypoints and descriptors for every single image
    vector<ImageFeatures> features(images.size());

    // Instantiate ORB (Oriented FAST and Rotated BRIEF), a fast and free feature extractor
    Ptr<Feature2D> finder = ORB::create();

    // Loop through each image and extract robust features (keypoints) to match overlapping areas
    for (size_t i = 0; i < images.size(); ++i) {
        computeImageFeatures(finder, images[i], features[i]);
    }

    // pairwise_matches will track which features in img A match features in img B
    vector<MatchesInfo> pairwise_matches;

    // Use BestOf2NearestMatcher to find correspondences between images. 0.3f is the match confidence threshold.
    Ptr<FeaturesMatcher> matcher = makePtr<BestOf2NearestMatcher>(false, 0.3f);

    // Perform the heavy N x N match computation between all features
    (*matcher)(features, pairwise_matches);

    // Free memory discarding low-confidence or unused match results
    matcher->collectGarbage();

    // ==========================================
    // STEP 2: CAMERA ESTIMATION & BUNDLE ADJUSTMENT
    // ==========================================
    cout << "Step 2: Camera Estimation and Bundle Adjustment..." << endl;

    // cameras will hold intrinsic (focal length) and extrinsic (rotation) params
    vector<CameraParams> cameras;

    // Estimate initial camera alignments based on how features move between matching image pairs
    Ptr<Estimator> estimator = makePtr<HomographyBasedEstimator>();
    if (!(*estimator)(features, pairwise_matches, cameras)) {
        cout << "Error: Homography estimation failed. Images may not overlap enough." << endl;
        return -1;
    }

    // Normalize camera rotation matrices to 32-bit floats for stability
    for (size_t i = 0; i < cameras.size(); ++i) {
        Mat R;
        cameras[i].R.convertTo(R, CV_32F);
        cameras[i].R = R;
    }

    // Refine the crude homography cameras using Bundle Adjustment (Ray logic minimizes ray projection error)
    Ptr<BundleAdjusterBase> adjuster = makePtr<BundleAdjusterRay>();
    adjuster->setConfThresh(1.0); // Strict confidence threshold

    if (!(*adjuster)(features, pairwise_matches, cameras)) {
        cout << "Error: Bundle Adjustment failed." << endl;
        return -1;
    }

    // Extract all focal lengths to determine the scale for our spherical projection canvas
    vector<double> focals;
    for (size_t i = 0; i < cameras.size(); ++i) focals.push_back(cameras[i].focal);
    sort(focals.begin(), focals.end());

    // Use the median focal length as the global scale for warping the images
    float warped_image_scale = static_cast<float>(focals[focals.size() / 2]);

    // ==========================================
    // COLAB FIX 2: MANUAL RAM DUMP
    // ==========================================
    cout << "Clearing intermediate RAM..." << endl;
    // Release massive data structures no longer needed after camera parameters are locked
    features.clear();
    pairwise_matches.clear();

    // ==========================================
    // STEP 3: SPHERICAL IMAGE WARPING
    // ==========================================
    cout << "Step 3: Warping to Spherical Canvas..." << endl;
    vector<Point> corners(images.size());         // Top-left coordinate of each image in the panorama
    vector<UMat> masks_warped(images.size());     // Indicates visible pixels for each warped image
    vector<UMat> images_warped(images.size());    // The curved/warped images
    vector<Size> sizes(images.size());            // Dimensions of warped bounding boxes
    vector<UMat> masks(images.size());            // Base masks (pure white before warping)

    // Create solid white masks the exact same size as the incoming images
    for (size_t i = 0; i < images.size(); ++i) {
        masks[i].create(images[i].size(), CV_8U);
        masks[i].setTo(Scalar::all(255));
    }

    // Setup the Spherical Warper (best for wide panoramas) mapping images onto a sphere
    Ptr<WarperCreator> warper_creator = makePtr<cv::SphericalWarper>();
    Ptr<RotationWarper> warper = warper_creator->create(warped_image_scale);

    for (size_t i = 0; i < images.size(); ++i) {
        // Prepare the camera intrinsic matrix K
        Mat_<float> K;
        cameras[i].K().convertTo(K, CV_32F);

        // Warp the actual image, outputting the deformed image and its top-left corner location on the canvas
        corners[i] = warper->warp(images[i], K, cameras[i].R, INTER_LINEAR, BORDER_REFLECT, images_warped[i]);
        sizes[i] = images_warped[i].size();

        // Warp the mask the exact same way so we know which pixels are valid image vs empty space
        warper->warp(masks[i], K, cameras[i].R, INTER_NEAREST, BORDER_CONSTANT, masks_warped[i]);
    }

    // ==========================================
    // STEP 4: EXPOSURE COMPENSATION & SEAM FINDING
    // ==========================================
    cout << "Step 4: Compensating Exposure and Finding Seams..." << endl;

    // Normalize brightness/exposure differences across overlapping regions so the sky doesn't patchily shift in brightness
    Ptr<ExposureCompensator> compensator = ExposureCompensator::createDefault(ExposureCompensator::GAIN_BLOCKS);
    compensator->feed(corners, images_warped, masks_warped);

    // ==========================================
    // COLAB FIX 3: LIGHTWEIGHT SEAM FINDER
    // ==========================================
    // Voronoi is lightning fast and uses almost 0 RAM compared to GraphCut.
    // Seam finding figures out exactly where to slice overlapping images to hide alignment ghosts.
    Ptr<SeamFinder> seam_finder = makePtr<detail::VoronoiSeamFinder>();

    // Voronoi expects floating point images representing gradients
    vector<UMat> images_warped_f(images.size());
    for (size_t i = 0; i < images.size(); ++i) {
        images_warped[i].convertTo(images_warped_f[i], CV_32F);
    }

    // The find() function modifies masks_warped in-place, slicing the masks where seams are optimal
    seam_finder->find(images_warped_f, corners, masks_warped);
    images_warped_f.clear(); // Free memory

    // ==========================================
    // STEP 5: MULTI-BAND BLENDING
    // ==========================================
    cout << "Step 5: Multi-Band Blending..." << endl;

    // Multi-band blender uses Laplacian pyramids to smooth low frequencies (colors) over wide ranges
    // and high frequencies (edges) over short ranges, preventing blur at seams.
    Ptr<Blender> blender = Blender::createDefault(Blender::MULTI_BAND);

    // Determine the size of the final composite canvas using corner bounds and sizes
    Rect dst_roi = resultRoi(corners, sizes);
    blender->prepare(dst_roi);

    // Feed each processed, warped, seamlessly sliced image into the blending pyramid
    for (size_t i = 0; i < images.size(); ++i) {
        // Apply the calculated exposure brightness adjustments
        compensator->apply(i, corners[i], images_warped[i], masks_warped[i]);

        // Blenders require 16-bit signed shorts to safely handle pyramid math without overflowing
        Mat img_warped_s;
        images_warped[i].convertTo(img_warped_s, CV_16S);
        Mat mask_warped_8u;
        masks_warped[i].convertTo(mask_warped_8u, CV_8U);

        // Add the image to the blender
        blender->feed(img_warped_s, mask_warped_8u, corners[i]);
    }

    // Export the finalized panorama and a mask defining valid pixel bounds on the canvas
    Mat result, result_mask;
    blender->blend(result, result_mask);

    // Convert back down to standard 8-bit color for saving to disk
    result.convertTo(result, CV_8U);
    imwrite(outputPath, result);

    cout << "Success! Panorama saved to: " << outputPath << endl;
    return 0;
}

Writing advanced_stitcher.cpp


In [23]:
!g++ advanced_stitcher.cpp -o app `pkg-config --cflags --libs opencv4`

In [24]:
# 2. Run the application 3 times.
# Format: !./app "<Input_Folder_Path>" "<Output_File_Path>"
!./app "/content/drive/MyDrive/MachineVision/Data/office2" "/content/drive/MyDrive/MachineVision/Output/panorama_advanced_1.jpg"


Loading and downscaling images for Colab compatibility...
Loaded 29 images.
Step 1: Finding and Matching Features (ORB)...
Step 2: Camera Estimation and Bundle Adjustment...
Clearing intermediate RAM...
Step 3: Warping to Spherical Canvas...
Step 4: Compensating Exposure and Finding Seams...
Step 5: Multi-Band Blending...
Success! Panorama saved to: /content/drive/MyDrive/MachineVision/Output/panorama_advanced_1.jpg


In [25]:
!./app "/content/drive/MyDrive/MachineVision/Data/WLH" "/content/drive/MyDrive/MachineVision/Output/panorama_advanced_2.jpg"


Loading and downscaling images for Colab compatibility...
Loaded 25 images.
Step 1: Finding and Matching Features (ORB)...
Step 2: Camera Estimation and Bundle Adjustment...
Clearing intermediate RAM...
Step 3: Warping to Spherical Canvas...
Step 4: Compensating Exposure and Finding Seams...
Step 5: Multi-Band Blending...
Success! Panorama saved to: /content/drive/MyDrive/MachineVision/Output/panorama_advanced_2.jpg


In [26]:
!./app "/content/drive/MyDrive/MachineVision/Data/StJames" "/content/drive/MyDrive/MachineVision/Output/panorama_advanced_3.jpg"


Loading and downscaling images for Colab compatibility...
Loaded 16 images.
Step 1: Finding and Matching Features (ORB)...
Step 2: Camera Estimation and Bundle Adjustment...
Clearing intermediate RAM...
Step 3: Warping to Spherical Canvas...
Step 4: Compensating Exposure and Finding Seams...
Step 5: Multi-Band Blending...
Success! Panorama saved to: /content/drive/MyDrive/MachineVision/Output/panorama_advanced_3.jpg


In [27]:
%%writefile basic-stitcher.cpp
#include <iostream>
#include <stdio.h>
#include <vector>
#include <math.h>
#include <opencv2/imgcodecs.hpp>
#include <opencv2/imgproc.hpp>
#include <opencv2/videoio.hpp>
#include <opencv2/highgui.hpp>
#include <opencv2/video.hpp>
#include <opencv2/calib3d.hpp>
#include <opencv2/opencv.hpp>
#include "opencv2/core/core.hpp"
#include <opencv2/core/utility.hpp>
#include "opencv2/features2d.hpp"

using namespace cv;
using namespace std;

// Struct to store an image along with its precomputed SIFT features.
// This prevents redundant computation when comparing the same image against multiple others.
struct ImageNode {
    Mat img;
    vector<KeyPoint> keypoints;
    Mat descriptors;
};

// Helper function to extract features once and cache them in ImageNode structures
vector<ImageNode> computeFeatures(const vector<Mat>& images) {
    // Instantiate the SIFT detector. We enforce a strict limit of 500 features per image.
    // This limit dramatically speeds up FLANN matching and Homography estimation later.
    Ptr<SIFT> detector = SIFT::create(500);

    // Allocate the output nodes list
    vector<ImageNode> nodes(images.size());

    for (size_t i = 0; i < images.size(); i++) {
        nodes[i].img = images[i];
        if(!images[i].empty()) {
            // detectAndCompute calculates both the pixel locations of corners (keypoints)
            // and their mathematical fingerprints (descriptors)
            detector->detectAndCompute(images[i], noArray(), nodes[i].keypoints, nodes[i].descriptors);
        }
    }
    return nodes;
}

// Function to crop out unnecessary solid black borders that accumulate during perspective warping
Mat trim(Mat image) {
    Mat result, grayImage, thresholded;
    // Convert to grayscale to evaluate pixel brightness
    cvtColor(image, grayImage, COLOR_BGR2GRAY);
    // Binarize the image: any pixel > 1 brightness becomes 255 (white). Absolute black (0) stays 0.
    threshold(grayImage, thresholded, 1, 255, THRESH_BINARY);

    vector<vector<Point>> contours;
    // Find the outlines of the non-black regions
    findContours(thresholded, contours, RETR_EXTERNAL, CHAIN_APPROX_SIMPLE);

    // Fallback if the image lacks distinct content
    if(contours.empty()) return image;

    // Compute the minimum bounding rectangle that contains all the valid pixels
    Rect rect = boundingRect(contours[0]);
    // Crop the image to exactly that bounding box
    return image(rect);
}

// Core function that physically stitches two images together using Homography
Mat imageStitch(ImageNode sceneNode, ImageNode objNode) {
    // Safety check: both images need at least 4 descriptors to compute a valid homography matrix
    if(objNode.descriptors.empty() || sceneNode.descriptors.empty() || objNode.descriptors.rows < 4 || sceneNode.descriptors.rows < 4)
        return sceneNode.img;

    // Use Fast Library for Approximate Nearest Neighbors (FLANN) for lightning-fast matching
    FlannBasedMatcher matcher;
    vector<DMatch> matches;

    // FLANN requires descriptors to be strictly in 32-bit floating point format
    if(objNode.descriptors.type() != CV_32F) objNode.descriptors.convertTo(objNode.descriptors, CV_32F);
    if(sceneNode.descriptors.type() != CV_32F) sceneNode.descriptors.convertTo(sceneNode.descriptors, CV_32F);

    // Compare descriptors and find the closest match for each feature
    matcher.match(objNode.descriptors, sceneNode.descriptors, matches);

    vector<Point2f> keypointsobj, keypointsscene;
    // Isolate the physical (x, y) coordinates of the matching feature pairs
    for(int i = 0; i < matches.size(); i++) {
        keypointsobj.push_back(objNode.keypoints[matches[i].queryIdx].pt);
        keypointsscene.push_back(sceneNode.keypoints[matches[i].trainIdx].pt);
    }

    // Guarantee enough points exist
    if(keypointsobj.size() < 4) return sceneNode.img;

    // Find the 3x3 transformation matrix defining how to deform 'obj' to perspective match 'scene'
    // RANSAC algorithm filters out wildly incorrect feature matches (outliers)
    Mat homography = findHomography(keypointsobj, keypointsscene, RANSAC);
    if(homography.empty()) return sceneNode.img;

    // Determine the corner coordinates of the incoming image
    vector<Point2f> obj_corners(4);
    obj_corners[0] = Point2f(0, 0);
    obj_corners[1] = Point2f((float)objNode.img.cols, 0);
    obj_corners[2] = Point2f((float)objNode.img.cols, (float)objNode.img.rows);
    obj_corners[3] = Point2f(0, (float)objNode.img.rows);

    // Predict where those 4 corners will land in the target 'scene' space after transformation
    vector<Point2f> obj_corners_transformed(4);
    perspectiveTransform(obj_corners, obj_corners_transformed, homography);

    // Determine the absolute boundaries of the joint canvas (combining both images)
    float min_x = 0, min_y = 0;
    float max_x = (float)sceneNode.img.cols;
    float max_y = (float)sceneNode.img.rows;

    for(int i = 0; i < 4; i++) {
        if (obj_corners_transformed[i].x < min_x) min_x = obj_corners_transformed[i].x;
        if (obj_corners_transformed[i].y < min_y) min_y = obj_corners_transformed[i].y;
        if (obj_corners_transformed[i].x > max_x) max_x = obj_corners_transformed[i].x;
        if (obj_corners_transformed[i].y > max_y) max_y = obj_corners_transformed[i].y;
    }

    // Generate an offset Translation matrix.
    // If transformed coordinates went negative, we shift everything right/down so they start at (0,0)
    Mat T = Mat::eye(3, 3, CV_64F);
    T.at<double>(0, 2) = -min_x;
    T.at<double>(1, 2) = -min_y;

    Size new_size((int)(max_x - min_x), (int)(max_y - min_y));

    // Safety check to prevent memory explosions caused by insane homography scaling
    if (new_size.width > 8000 || new_size.height > 8000) return sceneNode.img;

    // Warp the new image into the calculated combined canvas size
    Mat result;
    warpPerspective(objNode.img, result, T * homography, new_size);

    // Create a bounding box representing where the original 'scene' image belongs on the canvas
    Mat roi(result, Rect((int)-min_x, (int)-min_y, sceneNode.img.cols, sceneNode.img.rows));

    // Mask out the empty areas so we can securely overlay 'scene' on top of the warped 'obj'
    Mat mask;
    cvtColor(sceneNode.img, mask, COLOR_BGR2GRAY);
    threshold(mask, mask, 1, 255, THRESH_BINARY);

    // Paste the rigid 'scene' image rigidly on top, using the warped 'obj' as the background fill
    sceneNode.img.copyTo(roi, mask);

    // Crop excess black bordering
    return trim(result);
}

// Predicts which images heavily overlap by mapping out coordinate-validated (inlier) feature matches
Mat findInliersMatrix(vector<ImageNode> nodes) {
    // Output matrix where cell (i, j) = the number of reliable overlapping features between image i and image j
    Mat inlierMatrix = Mat::zeros(nodes.size(), nodes.size(), CV_64F);
    FlannBasedMatcher matcher;

    cout << "  Matching pairs ";
    // Compare every image against every other image
    for(int i = 0; i < nodes.size(); i++) {
        for(int j = 0 ; j < nodes.size(); j++) {
            if(i == j || nodes[i].descriptors.rows < 4 || nodes[j].descriptors.rows < 4) continue;

            // Ensure descriptors are CV_32F for FLANN
            if(nodes[i].descriptors.type() != CV_32F) nodes[i].descriptors.convertTo(nodes[i].descriptors, CV_32F);
            if(nodes[j].descriptors.type() != CV_32F) nodes[j].descriptors.convertTo(nodes[j].descriptors, CV_32F);

            // Match the two images' feature lists
            vector<DMatch> matches;
            matcher.match(nodes[i].descriptors, nodes[j].descriptors, matches);

            vector<Point2f> keypointsobj, keypointsscene;
            for(int k = 0; k < matches.size(); k++)  {
                keypointsobj.push_back(nodes[i].keypoints[matches[k].queryIdx].pt);
                keypointsscene.push_back(nodes[j].keypoints[matches[k].trainIdx].pt);
            }

            int inliersSize = 0;
            if (keypointsobj.size() >= 4) {
                vector<uchar> inliersStatus;
                // Calculate homography. inliersStatus will contain '1' for points that successfully conform
                // to the derived perspective matrix, and '0' for garbage matches.
                findHomography(keypointsobj, keypointsscene, RANSAC, 3, inliersStatus);
                for(int k = 0; k < inliersStatus.size(); k++) {
                    if(inliersStatus[k] == 1) inliersSize++;
                }
            }
            // Record the inlier score
            inlierMatrix.at<double>(i, j) = inliersSize;
        }
        cout << "." << flush; // Print a dot to show it hasn't frozen
    }
    cout << " Done!" << endl;
    return inlierMatrix;
}

// Executes a hierarchical "tournament style" stitching process
vector<Mat> pairwiseImageStitch(vector<ImageNode> nodes) {
    // Figure out which image pairs share the most visual data
    Mat inlierMatrix = findInliersMatrix(nodes);

    vector<int> imagesOrder;
    vector<bool> matched(nodes.size(), false);

    // Greedily lock-in pairs with the highest absolute connectivity.
    // For every image 'i', we find the single image 'j' that overlaps with it the absolute best.
    for(int i = 0; i < nodes.size(); i++) {
        if(matched[i]) continue;

        Mat row = inlierMatrix.row(i);
        double maxVal;
        Point maxLoc;
        // Search this row (image i's connections) for the peak overlap
        minMaxLoc(row, NULL, &maxVal, NULL, &maxLoc);

        int indexOfRowMax = maxLoc.x;

        // If there's an actual match and target isn't already grouped
        if(maxVal > 0 && !matched[indexOfRowMax]) {
            imagesOrder.push_back(i);
            imagesOrder.push_back(indexOfRowMax);
            matched[i] = true;
            matched[indexOfRowMax] = true;

            // Wipe data for both images so they can't be chosen again
            inlierMatrix.row(i).setTo(0);
            inlierMatrix.col(i).setTo(0);
            inlierMatrix.row(indexOfRowMax).setTo(0);
            inlierMatrix.col(indexOfRowMax).setTo(0);
        }
    }

    // Add any orphan images cleanly at the end so they aren't lost
    for(int i = 0; i < nodes.size(); i++) {
        if(!matched[i]) {
            imagesOrder.push_back(i);
        }
    }

    vector<Mat> stitchedImages;
    cout << "  Stitching ";

    // Run the physical stitching math on adjacent pairs
    for(int i = 0 ; i < imagesOrder.size() - 1; i += 2) {
        stitchedImages.push_back(imageStitch(nodes[imagesOrder[i]], nodes[imagesOrder[i+1]]));
        cout << "*" << flush; // Print a star for every successful stitch execution
    }

    // If an odd number of images existed, pass the last one directly to the next round unchanged
    if(imagesOrder.size() % 2 != 0) {
        stitchedImages.push_back(nodes[imagesOrder.back()].img);
    }
    cout << " Done!" << endl;

    // We reduced the array size by roughly half. This new array is returned for the next iteration.
    return stitchedImages;
}

int main(int argc, char** argv) {
    // Validate command-line arguments: directory of input images, and where to dump the panorama
    if (argc < 3) {
        cout << "Usage: ./app <input_folder_path> <output_file_path>" << endl;
        return -1;
    }

    String folderPath = argv[1];
    String outputPath = argv[2];
    String path = folderPath + "/*.jpg";

    vector<cv::String> result;
    vector<cv::Mat> images;

    // Recursively collect JPEG images from input folder
    cv::glob(path, result, false);

    // Load physical matrices into memory
    for (size_t k = 0; k < result.size(); ++k) {
        cv::Mat image = cv::imread(result[k]);
        if (image.empty()) continue;
        images.push_back(image);
    }

    cout << "Processing folder: " << folderPath << endl;
    cout << "Target Output: " << outputPath << endl;
    cout << "Number of input images: " << images.size() << endl;

    if(images.empty()) return -1;

    // Force every raw loaded image down to an 800x800 square to standardize extraction dimensions
    for(int i = 0; i < images.size(); i++) {
        resize(images[i], images[i], Size(800, 800));
    }

    vector<Mat> stitchedImages = images;
    int iteration = 1;

    // Run a while loop that hierarchically stitches N images down until 1 giant image remains
    while(stitchedImages.size() > 1) {
        cout << "--- Stitching Iteration " << iteration << " (Images: " << stitchedImages.size() << ") ---" << endl;

        // Cache the SIFT features of our current pool of sub-panoramas
        vector<ImageNode> currentNodes = computeFeatures(stitchedImages);
        // Execute a round of pairing, turning N images into ~N/2 images
        stitchedImages = pairwiseImageStitch(currentNodes);

        // RAM FIX 3: BULLETPROOF SCALING
        // Force every composite/stitched canvas back down to a maximum of 800px.
        // If we didn't do this, iteration 1 produces 1600px images, iteration 2 produces 3200px images,
        // which radically increases execution time exponentially and causes out-of-memory crashes.
        for(size_t i = 0; i < stitchedImages.size(); i++) {
            double max_dim = max(stitchedImages[i].cols, stitchedImages[i].rows);
            if (max_dim > 800) {
                double scale = 800.0 / max_dim;
                resize(stitchedImages[i], stitchedImages[i], Size(), scale, scale);
            }
        }

        iteration++;
    }

    // Only 1 image remains in the pool: our final panorama.
    if(stitchedImages.size() == 1) {
        cout << "Final Panorama Generated Successfully." << endl;
        imwrite(outputPath, stitchedImages[0]);
    }

    return 0;
}

Writing basic-stitcher.cpp


In [28]:
!g++ basic-stitcher.cpp -o app `pkg-config --cflags --libs opencv4`

In [29]:
!./app "/content/drive/MyDrive/MachineVision/Data/office2" "/content/drive/MyDrive/MachineVision/Output/panorama_basic_1.jpg"


Processing folder: /content/drive/MyDrive/MachineVision/Data/office2
Target Output: /content/drive/MyDrive/MachineVision/Output/panorama_basic_1.jpg
Number of input images: 29
--- Stitching Iteration 1 (Images: 29) ---
  Matching pairs ............................. Done!
  Stitching ************** Done!
--- Stitching Iteration 2 (Images: 15) ---
  Matching pairs ............... Done!
  Stitching ******* Done!
--- Stitching Iteration 3 (Images: 8) ---
  Matching pairs ........ Done!
  Stitching **** Done!
--- Stitching Iteration 4 (Images: 4) ---
  Matching pairs .... Done!
  Stitching ** Done!
--- Stitching Iteration 5 (Images: 2) ---
  Matching pairs .. Done!
  Stitching * Done!
Final Panorama Generated Successfully.


In [30]:
!./app "/content/drive/MyDrive/MachineVision/Data/WLH" "/content/drive/MyDrive/MachineVision/Output/panorama_basic_2.jpg"


Processing folder: /content/drive/MyDrive/MachineVision/Data/WLH
Target Output: /content/drive/MyDrive/MachineVision/Output/panorama_basic_2.jpg
Number of input images: 25
--- Stitching Iteration 1 (Images: 25) ---
  Matching pairs ......................... Done!
  Stitching ************ Done!
--- Stitching Iteration 2 (Images: 13) ---
  Matching pairs ............. Done!
  Stitching ****** Done!
--- Stitching Iteration 3 (Images: 7) ---
  Matching pairs ....... Done!
  Stitching *** Done!
--- Stitching Iteration 4 (Images: 4) ---
  Matching pairs .... Done!
  Stitching ** Done!
--- Stitching Iteration 5 (Images: 2) ---
  Matching pairs .. Done!
  Stitching * Done!
Final Panorama Generated Successfully.


In [31]:
!./app "/content/drive/MyDrive/MachineVision/Data/StJames" "/content/drive/MyDrive/MachineVision/Output/panorama_basic_3.jpg"


Processing folder: /content/drive/MyDrive/MachineVision/Data/StJames
Target Output: /content/drive/MyDrive/MachineVision/Output/panorama_basic_3.jpg
Number of input images: 16
--- Stitching Iteration 1 (Images: 16) ---
  Matching pairs ................ Done!
  Stitching ******** Done!
--- Stitching Iteration 2 (Images: 8) ---
  Matching pairs ........ Done!
  Stitching **** Done!
--- Stitching Iteration 3 (Images: 4) ---
  Matching pairs .... Done!
  Stitching ** Done!
--- Stitching Iteration 4 (Images: 2) ---
  Matching pairs .. Done!
  Stitching * Done!
Final Panorama Generated Successfully.
